# Rebuild the purchase report from raw events

*Data Analysis in Python · Lesson 7 final exercise*

The merchandising team wants Black Friday week's item revenue and units by item category, built from the raw GA4 events, and proof that it adds up to the store's revenue, before the Monday meeting.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the cell that downloads the file and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Create `records`: a list with one dictionary per line of `EVENTS_FILE`.
2. Add a `params` key to every dictionary in `records`: the event's `event_params` as a dictionary, with each parameter's `key` as its key and its `value` as its value. Then create the `events` table: `records` flattened, one row per event.
3. Create the `purchases` table: the rows of the `events` table whose `event_name` is `purchase`.
4. Create the `purchase_items` table: one row per item of the purchase events, made from `records`, with each event's `event_timestamp` and transaction ID as columns.
5. Create the `category_report` table: the `purchase_items` table totaled by `item_category`, with two columns: `item_revenue` and `quantity`.
6. Create `items_sold`: the number of units sold, from the `purchase_items` table.
7. Create `purchase_revenue`: the total of the purchase events' `value` parameter, from the `purchases` table. Then create `difference`: the `category_report` table's total item revenue minus `purchase_revenue`.
8. Add a text cell for the merchandising team: which item category brought the most revenue, and what share of the week's revenue was it? How many units sold, in how many items and purchases? Does the report add up to the purchases' revenue?
9. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `records` has 12,511 dictionaries, and the `events` table 12,511 rows with the parameters as columns. The `purchases` table has 150 rows. The `purchase_items` table has 217 rows, one per item, with `event_timestamp` and the transaction ID as columns. The `category_report` table has Women $6,479.78 and 152 units, Men $3,514.00 and 70, and Unisex $204.28 and 12. `items_sold` is `234`. `purchase_revenue` is about 10,198.06, and `difference` is about `0`.

## Parameters

In [ ]:
import json
import urllib.request

import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
EVENTS_FILE = "events_ga4_sample.ndjson"

In [ ]:
# Download the GA4 sample to this session's files. Run it, and leave it as it is.
urllib.request.urlretrieve(DATA_URL + EVENTS_FILE, EVENTS_FILE)
print(f"Downloaded {EVENTS_FILE}")

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    records = found.get("records")
    events = found.get("events")
    purchases = found.get("purchases")
    if not isinstance(records, list) or len(records) != 12511:
        results.append((False, "records should be a list of 12,511 dictionaries, one per line of the file: [json.loads(line) for line in file] (topic 7.1)."))
    elif not isinstance(events, pd.DataFrame) or len(events) != 12511:
        results.append((False, "events should be the records flattened, with 12,511 rows: pd.json_normalize(records), after you add params to every record."))
    elif "params.value.double_value" not in events.columns:
        results.append((False, "The events table has no params.value.double_value column. Add params to every record first: record[\"params\"] = {param[\"key\"]: param[\"value\"] for param in record[\"event_params\"]} (topic 7.2)."))
    elif not isinstance(purchases, pd.DataFrame) or len(purchases) != 150:
        results.append((False, "purchases should be the events table's 150 purchase rows: events[events[\"event_name\"] == \"purchase\"]."))
    else:
        results.append((True, "The events table has 12,511 events with their parameters as columns, and 150 of them are purchases."))

    items = found.get("purchase_items")
    report = found.get("category_report")
    items_sold = found.get("items_sold")
    if not isinstance(items, pd.DataFrame):
        results.append((False, "purchase_items should be a DataFrame: pd.json_normalize() with record_path=\"items\" on the purchase records (topic 7.2)."))
    elif "items" in items.columns:
        results.append((False, "The purchase_items table still has an items column. Make the rows from the list with record_path=\"items\", from the purchase records."))
    elif len(items) == 1799:
        results.append((False, "The purchase_items table has 1,799 rows: the items of every event, including view_item, add_to_cart and begin_checkout. Keep only the purchase records first."))
    elif len(items) == 456:
        results.append((False, "The purchase_items table has 456 rows: begin_checkout events carry the same order's items before the purchase. Keep only the purchase records."))
    elif len(items) != 217 or "item_revenue" not in items.columns:
        results.append((False, f"The purchase_items table should have 217 rows, one per item of the 150 purchases, with the items' columns such as item_revenue. It has {len(items):,} rows."))
    elif "event_timestamp" not in items.columns or not any("transaction_id" in str(column) for column in items.columns):
        results.append((False, f"The purchase_items table should have event_timestamp and the transaction ID as columns: meta=[\"event_timestamp\", [\"params\", \"transaction_id\", \"string_value\"]], or [\"ecommerce\", \"transaction_id\"]. It has {list(items.columns)}."))
    elif not isinstance(report, pd.DataFrame) or not {"item_revenue", "quantity"} <= set(report.columns):
        results.append((False, "category_report should be a DataFrame with the columns item_revenue and quantity: purchase_items.groupby(\"item_category\")[[\"item_revenue\", \"quantity\"]].sum()."))
    elif "Pants" in report.index:
        results.append((False, "The category_report table is by item_category2, the product category. The brief asks for item_category: Women, Men and Unisex."))
    elif set(report.index) != {"Women", "Men", "Unisex"}:
        results.append((False, f"The category_report table should have one row per item_category: Women, Men and Unisex. Its index is {list(report.index)}."))
    elif abs(report.loc["Women", "item_revenue"] - 5887.01) < 0.01:
        results.append((False, "The category_report table's item_revenue for Women is $5,887.01: that's the price, once per item row. Some items were bought two at a time: total item_revenue, which is price times quantity."))
    elif report.loc["Women", "quantity"] == 139:
        results.append((False, "The category_report table's quantity for Women is 139: that's the number of item rows. Total the quantity column, because some rows are two units."))
    elif abs(report.loc["Women", "item_revenue"] - 6479.78) > 0.01 or abs(report.loc["Men", "item_revenue"] - 3514.00) > 0.01 or abs(report.loc["Unisex", "item_revenue"] - 204.28) > 0.01:
        results.append((False, f"The category_report table's item_revenue should be Women $6,479.78, Men $3,514.00 and Unisex $204.28: the purchase_items table's item_revenue, totaled. It is {report['item_revenue'].round(2).to_dict()}."))
    elif report["quantity"].to_dict() != {"Men": 70, "Unisex": 12, "Women": 152}:
        results.append((False, f"The category_report table's quantity should be Women 152, Men 70 and Unisex 12. It is {report['quantity'].to_dict()}."))
    elif not number(items_sold):
        results.append((False, f"items_sold should be a number, the units sold. It is {items_sold!r}."))
    elif items_sold == 217:
        results.append((False, "items_sold is 217: that's the purchase_items table's rows. Some rows are two units of the same product: total the quantity column."))
    elif items_sold == 150:
        results.append((False, "items_sold is 150: that's the purchase events. Count units: the purchase_items table's quantity, totaled."))
    elif items_sold != 234:
        results.append((False, f"items_sold should be 234, the purchase_items table's quantity totaled. It is {items_sold!r}."))
    else:
        top = report["item_revenue"].idxmax()
        results.append((True, f"{items_sold} units in 217 items and 150 purchases. {top} brought ${report.loc[top, 'item_revenue']:,.2f} of item revenue, {report.loc[top, 'item_revenue'] / report['item_revenue'].sum():.1%} of the week's."))

    revenue = found.get("purchase_revenue")
    difference = found.get("difference")
    if not number(revenue):
        results.append((False, f"purchase_revenue should be a number: the purchases table's params.value.double_value, totaled. It is {revenue!r}."))
    elif abs(revenue - 19099.34) < 0.01:
        results.append((False, "purchase_revenue is $19,099.34: each purchase's revenue, copied onto every one of its item rows by meta, and totaled. A purchase with three items counts three times. Total the purchases table's params.value.double_value, one value per purchase."))
    elif abs(revenue - 11634.75) < 0.01:
        results.append((False, "purchase_revenue is $11,634.75: the value plus tax and shipping. Item revenue doesn't include either, so compare it with the value parameter alone."))
    elif abs(revenue - 92361.63) < 0.01:
        results.append((False, "purchase_revenue is $92,361.63: the value parameter of every event that has one, including view_item and add_to_cart. Total it for the purchases table."))
    elif abs(revenue - 10198.06) > 0.01:
        results.append((False, f"purchase_revenue should be the purchases table's params.value.double_value, totaled: about 10,198.06. It is {revenue:,.2f}."))
    elif not number(difference):
        results.append((False, f"difference should be a number: category_report[\"item_revenue\"].sum() - purchase_revenue. It is {difference!r}."))
    elif abs(difference) > 0.01:
        results.append((False, f"difference should be about 0: the category_report table's total item_revenue minus purchase_revenue. It is {difference:,.2f}."))
    else:
        results.append((True, "The item report reconciles: $10,198.06 of item revenue, and $10,198.06 of purchase revenue, with a difference of $0.00."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Read your text cell again: it's what the merchandising team will read.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 1: one dictionary per event</b></summary>

Topic 7.1.: the file has one JSON object per line, so read it line by line.

~~~python
with open(EVENTS_FILE, encoding="utf-8") as file:
    records = [json.loads(...) for line in file]
len(records)
~~~

</details>

<details>
<summary><b>Step 2: the parameters as columns</b></summary>

Topic 7.2.: a dictionary comprehension turns each event's list of parameters into a dictionary. Store it in the record, then flatten all of them.

~~~python
for record in records:
    record["params"] = {param[...]: param[...] for param in record["event_params"]}
events = pd.json_normalize(...)
events.shape
~~~

</details>

<details>
<summary><b>Step 3: the purchases</b></summary>

Topic 7.1.: a filter on `event_name`.

~~~python
purchases = events[...]
len(purchases)
~~~

</details>

<details>
<summary><b>Step 4: one row per item</b></summary>

Topic 7.2.: keep the purchase records, then make rows from their `items` with `record_path`, and bring the event's values along with `meta`. A nested value in `meta` is a list of keys: the transaction ID is in the `params` you added, or in `ecommerce`.

~~~python
purchase_records = [record for record in records if ...]
purchase_items = pd.json_normalize(purchase_records, record_path=..., meta=["event_timestamp", [..., ...]])
purchase_items.shape
~~~

</details>

<details>
<summary><b>Step 5: the report</b></summary>

Lesson 4.: total two columns by group. Each item row has its own `item_revenue`: the price times the quantity of that product. The purchase's revenue isn't an item's, so leave it out of the report.

~~~python
category_report = purchase_items.groupby(...)[[..., ...]].sum()
category_report
~~~

</details>

<details>
<summary><b>Step 6: units sold</b></summary>

Topic 7.1.: a row of `purchase_items` is an item, not a purchase, and an item can have a quantity of 2.

~~~python
items_sold = purchase_items[...].sum()
~~~

</details>

<details>
<summary><b>Step 7: does it add up?</b></summary>

Topic 7.2.: total the money at both grains and compare. The purchase's revenue is in its `value` parameter, one value per purchase event.

~~~python
purchase_revenue = purchases[...].sum()
difference = category_report[...].sum() - purchase_revenue
purchase_revenue, difference
~~~

</details>

<details>
<summary><b>Step 8: the text cell</b></summary>

Choose **Insert → Text cell**. Give each category's revenue and its share of the total, the units, items and purchases, and say whether the report reconciles, and what the revenue doesn't include: look at the purchases' other parameters.

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
with open(EVENTS_FILE, encoding="utf-8") as file:
    records = [json.loads(line) for line in file]
len(records)
```

```python
for record in records:
    record["params"] = {param["key"]: param["value"] for param in record["event_params"]}
events = pd.json_normalize(records)
purchases = events[events["event_name"] == "purchase"]
events.shape, len(purchases)
```

```python
purchase_records = [record for record in records if record["event_name"] == "purchase"]
purchase_items = pd.json_normalize(purchase_records, record_path="items", meta=["event_timestamp", ["params", "transaction_id", "string_value"]])
assert len(purchase_items) == purchases["items"].str.len().sum()
purchase_items.shape
```

```python
category_report = purchase_items.groupby("item_category")[["item_revenue", "quantity"]].sum()
items_sold = purchase_items["quantity"].sum()
category_report, items_sold
```

```python
purchase_revenue = purchases["params.value.double_value"].sum()
difference = category_report["item_revenue"].sum() - purchase_revenue
purchase_revenue, difference
```

```markdown
**For the merchandising team.** Black Friday week 2025's 150 purchases had 217 items and 234 units, and $10,198.06 of item revenue:

- Women: $6,479.78, 63.5% of the week's, from 152 units
- Men: $3,514.00, 34.5%, from 70 units
- Unisex: $204.28, 2.0%, from 12 units

The report adds up: the items' revenue is the same as the purchases' revenue, $10,198.06, to the cent. Neither includes tax ($713.89) or shipping ($722.80).
```

</details>